# MP3: Shakespearebot
Language Modeling with LSTMs and Transformers

In this mini-project you will train character- and token-level language models on Shakespeare's plays and sonnets. You will work with three architectures of increasing scale:

| Part | Model | Data | Tokenization |
|------|-------|-------|-------------|
| **1** |  LSTM | Tiny Shakespeare → fine-tune on sonnets | Character (one-hot) |
| **2** | nanoGPT | Tiny Shakespeare → fine-tune on sonnets | SentencePiece BPE (1024 vocab) |
| **3** | Pretrained nanoGPT 124M | Fine-tune on sonnets  | GPT-2 BPE (50257 vocab) |

**Runtime:** Parts 1 and 2 train from scratch and benefit from a GPU/TPU runtime. Part 3 loads a pretrained model and fine-tunes it.


In [ ]:
# for tpu support.
# !pip install torch_xla[tpu]

In [ ]:
import torch

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    DEVICE = torch_xla.device()
    DEVICE_TYPE = 'tpu'
except ImportError:
    if torch.cuda.is_available():
        DEVICE = torch.device('cuda')
        DEVICE_TYPE = 'cuda'
    else:
        DEVICE = torch.device('cpu')
        DEVICE_TYPE = 'cpu'

if DEVICE_TYPE == 'tpu':
    dtype = 'bfloat16'
elif DEVICE_TYPE == 'cuda' and torch.cuda.is_bf16_supported():
    dtype = 'bfloat16'
elif DEVICE_TYPE == 'cuda':
    dtype = 'float16'
else:
    dtype = 'float32'

ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

print(f'DEVICE: {DEVICE_TYPE} - {DEVICE}')

#### Load in Data

We use two Shakespeare corpora:
- Tiny Shakespeare (~1.1M characters) — all plays, used for pre-training. We've provided other corpora in case you want to train on more data.
- Shakespeare Sonnets (~95K characters) — 154 sonnets, used for fine-tuning.

The sonnet file is preprocessed so that each sonnet begins with a special `@` marker, which the model can learn to use as a "start-of-sonnet" token during generation.

In [ ]:
import requests
import os

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
full_shakespeare_file_path = 'data/shakespeare_char/input.txt'
if not os.path.exists(full_shakespeare_file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    os.makedirs(os.path.dirname(full_shakespeare_file_path), exist_ok=True)
    with open(full_shakespeare_file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(data_url).text)

sonnet_file_path = 'data/shakespeare_sonnet/input.txt'
if not os.path.exists(sonnet_file_path):
    data_url = 'https://caltech-cs155.s3.us-east-2.amazonaws.com/miniprojects/project3/data/shakespeare.txt'
    os.makedirs(os.path.dirname(sonnet_file_path), exist_ok=True)
    with open(sonnet_file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(data_url).text)

In [ ]:
import re

with open(full_shakespeare_file_path, 'r', encoding='utf-8') as f:
    text = f.read()

with open(sonnet_file_path, 'r', encoding='utf-8') as f:
    sonnet_text_raw = f.read()

SONNET_START = '@'
sonnet_text_raw = re.sub(r'^ *\d+\n', SONNET_START + '\n', sonnet_text_raw, flags=re.MULTILINE)
print(sonnet_text_raw[:200])

---

### 1. Character-Level LSTM

This section builds a character-level language model using an LSTM (Long Short-Term Memory) network, following Andrej Karpathy's [blog on RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/).


#### 1.1 Tokenization

In [ ]:
chars = tuple(sorted(set(text) | set(sonnet_text_raw)))
itos = dict(enumerate(chars))
stoi = {ch: ii for ii, ch in itos.items()}
vocab_size = len(chars)

encoded = np.array([stoi[ch] for ch in text])

print(f"Dataset size: {len(text):,} charas")
print(f"Sonnet size: {len(sonnet_text_raw):,} chars")
print(f"Vocab size:  {vocab_size}")
print(f"Characters:  {repr(''.join(chars))}")

The text is split 90/10 into train/val for monitoring overfitting.

In [ ]:
n = len(encoded)
split_idx = int(n * 0.9)
train_data = encoded[:split_idx]
val_data = encoded[split_idx:]

print(f"Train: {len(train_data):,} tokens ({len(train_data)/n:.0%})")
print(f"Val: {len(val_data):,} tokens ({len(val_data)/n:.0%})")

#### 1.2 Batching

In [ ]:
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = len(arr)//batch_size_total

    # Trim to exact multiple so reshape works cleanly
    arr = arr[:n_batches * batch_size_total]
    # Reshape into `batch_size` parallel streams: each row is one
    # continuous thread of text, so hidden state carries across batches
    arr = arr.reshape((batch_size, -1))

    for n in range(0, arr.shape[1], seq_length):
        x = arr[:, n:n+seq_length]
        # y is x shifted right by one char (next-char prediction target)
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            # Wrap around to start of text at the very end
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

#### 1.3 Model

In [ ]:
class CharLSTM(nn.Module):

    def __init__(self, tokens, n_hidden=256, n_layers=2,
                               drop_prob=0.5, lr=0.001):
        super().__init__()
        self.drop_prob = drop_prob
        self.n_layers = n_layers
        self.n_hidden = n_hidden
        self.lr = lr

        # Build char int lookup tables from the full vocabulary
        self.chars = tokens
        self.itos = dict(enumerate(self.chars))
        self.stoi = {ch: ii for ii, ch in self.itos.items()}

        # Input size = vocab size because we feed one-hot vectors
        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layers,
                            dropout=drop_prob, batch_first=True)

        self.dropout = nn.Dropout(drop_prob)
        # Project hidden state back to vocab size for next-char logits
        self.fc = nn.Linear(n_hidden, len(self.chars))


    def forward(self, x, hidden):
        # x: (batch, seq_len, vocab_size) one-hot; hidden: (h, c) tuple
        r_output, hidden = self.lstm(x, hidden)
        out = self.dropout(r_output)
        # Flatten batch and seq dims so fc sees (batch*seq_len, n_hidden)
        out = out.contiguous().view(-1, self.n_hidden)
        out = self.fc(out)

        return out, hidden


    def init_hidden(self, batch_size):
        # Allocate zero hidden/cell states on the same device as model weights
        weight = next(self.parameters()).data
        hidden = (weight.new(self.n_layers, batch_size, self.n_hidden).zero_().to(DEVICE),
                  weight.new(self.n_layers, batch_size, self.n_hidden).zero_().to(DEVICE))
        return hidden


#### 1.4 Training function


In [ ]:
import time

def train(model, train_data, val_data, save_path, epochs=10, batch_size=10, seq_length=50, lr=0.001, clip=5, print_every=10):
    model.to(DEVICE)
    model.train()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    n_chars = len(model.chars)

    counter = 0
    best_val_loss = float('inf')
    train_loss_history, val_loss_history, steps = [], [], []
    t0 = time.time()

    for e in range(epochs):
        h = model.init_hidden(batch_size)

        for x, y in get_batches(train_data, batch_size, seq_length):
            counter += 1
            # One-hot encode input chars; flatten targets to (batch*seq,)
            inputs = F.one_hot(torch.tensor(x), num_classes=n_chars).float().to(DEVICE)
            targets = torch.tensor(y, dtype=torch.long, device=DEVICE).view(-1)
            # Detach hidden state from previous batch's graph to avoid BPTT
            # across the entire epoch (would blow up memory)
            h = tuple(each.data for each in h)

            optimizer.zero_grad()
            output, h = model(inputs, h)
            loss = criterion(output, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            if DEVICE_TYPE == 'tpu':
              xm.optimizer_step(optimizer)
            else:
              optimizer.step()

            if counter % print_every == 0:
                model.eval()
                val_h = model.init_hidden(batch_size)
                val_losses = []
                with torch.no_grad():
                    for vx, vy in get_batches(val_data, batch_size, seq_length):
                        vinputs = F.one_hot(torch.tensor(vx), num_classes=n_chars).float().to(DEVICE)
                        vtargets = torch.tensor(vy, dtype=torch.long, device=DEVICE).view(-1)
                        val_h = tuple(each.data for each in val_h)
                        vout, val_h = model(vinputs, val_h)
                        val_losses.append(criterion(vout, vtargets).item())
                model.train()

                avg_val = np.mean(val_losses)
                train_loss_history.append(loss.item())
                val_loss_history.append(avg_val)
                steps.append(counter)

                marker = ""
                if avg_val < best_val_loss:
                    best_val_loss = avg_val
                    torch.save({
                        'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict(),
                        'chars': model.chars,
                        'n_hidden': model.n_hidden,
                        'n_layers': model.n_layers,
                        'best_val_loss': best_val_loss,
                        'epoch': e,
                        'step': counter,
                    }, save_path)
                    marker = " *"

                elapsed = time.time() - t0
                print(f"Epoch {e+1}/{epochs} | Step {counter} | "
                      f"train {loss.item():.4f} | val {avg_val:.4f} | "
                      f"{elapsed:.1f}s{marker}")
                t0 = time.time()

    print(f"\nDone. Best val loss: {best_val_loss:.4f} — saved to {save_path}")
    return steps, train_loss_history, val_loss_history

#### 1.5 Hyperparameters and model instantiation

Fill in the `#TODO` values below. Use the tables as a starting point and experiment.

Training hyperparameters:

| Parameter | recommended | Notes |
|-----------|------------------|-------|
| `batch_size` | 32–128 | Number of parallel text streams. Higher = faster training and smoother gradients, but more GPU memory. |
| `seq_length` | 50–200 | Characters per training window. Longer = more context for the LSTM, but slower. |
| `n_epochs` | 10–30 | Full passes through the training data. Start with 20; |
| `lr` | 0.001–0.003 | Adam learning rate. |

Architecture hyperparameters:

| Parameter | recommended range | Notes |
|-----------|------------------|-------|
| `n_hidden` | 256–512 | LSTM hidden-state dimension. |
| `n_layers` | 2–3 | Stacked LSTM depth. |

Aim for a model with ~800K–6M total parameters for this dataset size.

In [ ]:
batch_size = #TODO
seq_length = #TODO
n_epochs = #TODO
lr = #TODO
clip=5
print_every=10

BEST_LSTM_PATH = 'out/best_lstm.pt'

In [ ]:
n_hidden= #TODO
n_layers= #TODO

model = CharLSTM(chars, n_hidden, n_layers)
print(model)

print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

#### 1.6 Training loop

In [ ]:
steps, train_loss_history, val_loss_history = train(
    model, train_data, val_data,
    save_path=BEST_LSTM_PATH,
    epochs=n_epochs, batch_size=batch_size,
    seq_length=seq_length, lr=lr, clip=clip,
    print_every=print_every,
)

#### 1.8 Loss curves

Plot train vs. validation loss to diagnose model behavior:

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(10, 5))
plt.plot(steps, train_loss_history, label='Train Loss', marker='o', markersize=3)
plt.plot(steps, val_loss_history, label='Val Loss', marker='s', markersize=3)
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('charLSTM: train vs validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### 1.9 Sampling functions

Two functions power text generation:

- `predict(model, char, h, top_k)`: feeds one character through the model and returns the predicted next character plus the updated hidden state.
- `sample(model, size, prime, top_k)`: generates `size` characters by repeatedly calling `predict`. The `prime` string "warms up" the hidden state before generation begins.

**`top_k`** limits sampling to the `k` most probable next characters. Lower values (e.g. 5) produce safer, more repetitive text; higher values (e.g. 20–40) allow more variety but risk incoherence. Set to `None` to sample from the full distribution.

In [ ]:
def predict(model, char, h=None, top_k=None):
        # Encode the single character as a one-hot vector
        x = torch.tensor([[model.stoi[char]]])
        inputs = F.one_hot(x, num_classes=len(model.chars)).float().to(DEVICE)

        h = tuple([each.data for each in h])
        out, h = model(inputs, h)

        # Convert logits → probabilities
        p = F.softmax(out, dim=1).data.cpu()

        if top_k is None:
            top_ch = np.arange(len(model.chars))
        else:
            # Keep only the top_k most likely characters
            p, top_ch = p.topk(top_k)
            top_ch = top_ch.numpy().squeeze()

        # Re-normalize and sample one character
        p = p.numpy().squeeze()
        char = np.random.choice(top_ch, p=p/p.sum())

        return model.itos[char], h

In [ ]:
def sample(model, size, prime='The', top_k=None):
    model.to(DEVICE)
    model.eval()

    chars = [ch for ch in prime]
    h = model.init_hidden(1)
    # "Prime" the hidden state by feeding the prompt one char at a time
    # (we discard all predictions except the last)
    for ch in prime:
        char, h = predict(model, ch, h, top_k=top_k)

    chars.append(char)

    # Autoregressive generation: feed each predicted char back in
    for ii in range(size):
        char, h = predict(model, chars[-1], h, top_k=top_k)
        chars.append(char)

    return ''.join(chars)

#### 1.10 Fine-tune LSTM on Shakespeare Sonnets

We load the best play-trained checkpoint and continue training on the much smaller sonnet corpus (~95K characters). This is transfer learning: the model already understands English character patterns and now adapts to the specific style and structure of sonnets.

Overfitting risk: The sonnet corpus is small, so the model can memorize it. Watch the val loss — if it rises while train loss keeps dropping.

In [ ]:
sonnet_encoded = np.array([stoi[ch] for ch in sonnet_text_raw])
sn = len(sonnet_encoded)
sonnet_split = int(sn * 0.9)
sonnet_train = sonnet_encoded[:sonnet_split]
sonnet_val = sonnet_encoded[sonnet_split:]

print(f"Sonnet dataset: {len(sonnet_text_raw):,} characters")
print(f"Vocab size:   {vocab_size} (same combined vocab)")
print(f"Train: {len(sonnet_train):,} tokens | Val: {len(sonnet_val):,} tokens")

In [ ]:
checkpoint = torch.load(BEST_LSTM_PATH, map_location=DEVICE, weights_only=False)
print(f"Loaded best checkpoint — val loss {checkpoint['best_val_loss']:.4f} "
      f"(epoch {checkpoint['epoch']+1}, step {checkpoint['step']})")

ft_model = CharLSTM(chars,
                  n_hidden=checkpoint['n_hidden'],
                  n_layers=checkpoint['n_layers'])
ft_model.load_state_dict(checkpoint['model_state'])
ft_model.to(DEVICE)

print(f"Fine-tune model: {sum(p.numel() for p in ft_model.parameters()):,} parameters")

In [ ]:
steps, train_data_losses, val_data_losses = train(
    ft_model, sonnet_train, sonnet_val,
    save_path='out/best_lstm_sonnet.pt',
    epochs=#TODO,
    batch_size=#TODO,
    seq_length=#TODO,
    lr=#TODO,
    print_every=5,
)

#### 1.11 Fine-tuning Loss Curve (Sonnets)

Compare with the pre-training loss curve. Fine-tuning typically starts at a lower loss (since the model is already trained) and converges faster.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(10, 5))
plt.plot(steps, train_data_losses, label='Train Loss', marker='o', markersize=3)
plt.plot(steps, val_data_losses, label='Val Loss', marker='s', markersize=3)
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('CharLSTM Fine-tuning on Sonnets: Train vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### 1.12 Sample Sonnets from Fine-tuned LSTM

Generate sonnets from the fine-tuned model. Start with the `@` sonnet marker so the model knows to write in sonnet style. Adding a few words steers the opening line.

In [ ]:
print(sample(ft_model, 500,
             prime=#TODO
             top_k=#TODO
             ))

---

### 2. nanoGPT (from Scratch)

We now train a standard Transformer GPT (following [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT)) on the same Shakespeare dataset.

- Tokenization: BPE (byte-pair encoding) via SentencePiece instead of single characters.
- Architecture: Multi-head self-attention + feedforward layers, with learned positional embeddings.


In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

import os
import math
import time
import inspect
from dataclasses import dataclass
from contextlib import nullcontext

#### 2.1 Model definition


| Parameter | Recommended range | Notes |
|-----------|------------------|-------|
| `block_size` | 128–256 | Maximum context length in BPE tokens. Must match the `block_size` used during data loading. |
| `vocab_size` | (set automatically) | Must equal the SentencePiece vocabulary size (1024 if you used `SP_VOCAB_SIZE = 1024`). |
| `n_layer` | 4–12 | Number of Transformer blocks. |
| `n_head` | 4–12 | Number of attention heads. Must evenly divide `n_embd`. 6 heads with n_embd=384 works well. |
| `n_embd` | 192–384 | Embedding dimension. Determines model width. |
| `dropout` | 0.1–0.3 | Applied after attention, MLP, and embeddings. |

With `n_layer=6, n_head=6, n_embd=384` and BPE vocab 1024, you get ~10M parameters which is a good balance between capacity and training speed for this dataset.

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = #TODO
    vocab_size: int = #TODO
    n_layer: int = #TODO
    n_head: int = #TODO
    n_embd: int = #TODO
    dropout: float = #TODO
    bias: bool = True

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Single linear projects input to Q, K, V concatenated (3x width)
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # Lower-triangular mask ensures position i can only attend to positions ≤ i
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        # Split the 3*n_embd projection into separate Q, K, V
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # Reshape into (B, n_head, T, head_dim) for multi-head attention
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # Scaled dot-product attention with causal mask
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        # Recombine heads and project back to n_embd
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    """Position-wise feedforward: expand 4x, GELU, project back."""
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class TransformerBlock(nn.Module):
    """Pre-norm Transformer block: LN → Attention → residual, LN → MLP → residual."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
class NanoGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight tying: share embedding and output projection weights
        # (reduces params and improves generalization — standard in GPT-2)
        self.token_emb.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.drop(self.token_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        # Only apply weight decay to 2D+ params (weight matrices), not biases/norms
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, fused=use_fused)
        return optimizer

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            # Crop context to block_size if it's grown too long
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            # Scale logits by temperature before softmax
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                # Zero out everything below the top-k threshold
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

#### 2.2 Tokenization (BPE)

In [ ]:
import numpy as np
import torch

import sentencepiece as spm

SP_PREFIX = 'out/sp_shakespeare'
SP_VOCAB_SIZE = 1024

spm.SentencePieceTrainer.train(
    input='data/shakespeare_char/input.txt',
    model_prefix=SP_PREFIX,
    vocab_size=SP_VOCAB_SIZE,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=3,
)

sp = spm.SentencePieceProcessor(model_file=f'{SP_PREFIX}.model')
vocab_size = sp.get_piece_size()

encode = lambda s: sp.encode(s)
decode = lambda ids: sp.decode(ids)

encoded = np.array(encode(text), dtype=np.int64)

n = len(encoded)
train_data = encoded[:int(n * 0.9)]
val_data = encoded[int(n * 0.9):]

print(f"BPE vocab size: {vocab_size}")
print(f"Text: {len(text):,} chars - {n:,} tokens ({len(text)/n:.1f} chars/token)")
print(f"Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens")

#### 2.3 Build model and data loader

In [ ]:
block_size = #TODO
batch_size = #TODO

BEST_GPT_PATH = 'out/best_nanogpt.pt'

config = GPTConfig(
    block_size=block_size,
    vocab_size=vocab_size,
    n_layer=#TODO,
    n_head=#TODO,
    n_embd=#TODO,
    dropout=#TODO,
    bias=False,
)

model = NanoGPT(config).to(DEVICE)
print(f"nanoGPT model: {model.get_num_params():,} parameters")

def gpt_get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].copy()) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].copy()) for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

Learning rates and training configuration:

| Parameter | Recommended range | Notes |
|-----------|------------------|-------|
| `learning_rate` | 5e-4 – 1e-3 | Peak LR. 6e-4 or 1e-3 are common for small Transformers. |
| `max_iters` | 3000–5000 | Total training steps. 5000 is a good target; monitor val loss to decide. |
| `warmup_iters` | 100–300 | Steps to ramp up LR. ~5% of `max_iters` is typical. |
| `lr_decay_iters` | ≈ `max_iters` | Usually set equal to `max_iters` so the cosine decay spans the full run. |
| `min_lr` | `learning_rate / 10` | Floor of the cosine schedule. E.g., if `learning_rate=6e-4`, use `min_lr=6e-5`. |

In [ ]:
learning_rate = #TODO
max_iters = #TODO
warmup_iters = #TODO
lr_decay_iters = #TODO
min_lr = #TODO
eval_interval = 500
eval_iters = 200
grad_clip = 1.0
log_interval = 100

ctx = torch.amp.autocast(device_type='cuda', dtype=ptdtype) if DEVICE_TYPE == 'cuda' else nullcontext()

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE_TYPE == 'cuda' and dtype == 'float16'))

#### 2.4 Training loop

In [ ]:
optimizer = model.configure_optimizers(
    weight_decay=1e-1,
    learning_rate=learning_rate,
    betas=(0.9, 0.99),
    device_type=DEVICE_TYPE,
)

def get_lr(it):
    # Phase 1: linear warmup from 0 → learning_rate
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    # Phase 3: constant at min_lr after decay is done
    if it > lr_decay_iters:
        return min_lr
    # Phase 2: cosine decay from learning_rate → min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = gpt_get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
torch.manual_seed(1337)

train_losses = []
val_losses = []
eval_steps = []
best_val_loss = 1e9

model.train()
t0 = time.time()

for iter_num in range(max_iters + 1):
    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        eval_steps.append(iter_num)
        marker = ""
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save({
                'model_state': model.state_dict(),
                'config': config,
                'best_val_loss': best_val_loss,
                'iter_num': iter_num,
            }, BEST_GPT_PATH)
            marker = " *"
        print(f"step {iter_num:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | lr {lr:.2e}{marker}")

    X, Y = gpt_get_batch('train')
    # Mixed-precision forward pass (float16/bfloat16 on GPU, no-op on CPU)
    with ctx:
        _, loss = model(X, Y)

    # GradScaler handles float16 loss scaling to prevent underflow
    scaler.scale(loss).backward()
    if grad_clip != 0.0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if iter_num % log_interval == 0 and iter_num > 0 and iter_num % eval_interval != 0:
        dt = time.time() - t0
        print(f"step {iter_num:5d} | loss {loss.item():.4f} | time {dt*1000:.0f}ms")
    t0 = time.time()

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

#### 2.5 Loss curves


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(eval_steps, train_losses, label='Train Loss', marker='o', markersize=3)
plt.plot(eval_steps, val_losses, label='Val Loss', marker='s', markersize=3)
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('(nanoGPT): Train vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

gap = val_losses[-1] - train_losses[-1]
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")
print(f"Gap (val - train): {gap:.4f}")

#### 2.7 Fine-tune nanoGPT on Shakespeare Sonnets

Load the best play-trained checkpoint and continue training on the sonnet corpus. so we need:
- Fewer iterations than pre-training.
- A lower learning rate to avoid catastrophic forgetting of general English.
- More frequent evaluation to catch overfitting early.


In [ ]:
checkpoint = torch.load(BEST_GPT_PATH, map_location=DEVICE, weights_only=False)
print(f"Loaded best play-trained checkpoint — val loss {checkpoint['best_val_loss']:.4f} "
      f"(step {checkpoint['iter_num']})")

In [ ]:
model = NanoGPT(config).to(DEVICE)
model.load_state_dict(checkpoint['model_state'])
print(f"Model: {model.get_num_params():,} parameters")

In [ ]:
sonnet_encoded = np.array(encode(sonnet_text_raw), dtype=np.int64)
sn = len(sonnet_encoded)
sonnet_train = sonnet_encoded[:int(sn * 0.9)]
sonnet_val = sonnet_encoded[int(sn * 0.9):]
print(f"Sonnets: {len(sonnet_text_raw):,} chars -> {sn:,} BPE tokens ({len(sonnet_text_raw)/sn:.1f} chars/token)")
print(f"Train: {len(sonnet_train):,} | Val: {len(sonnet_val):,}")

In [ ]:
ft_batch_size = #TODO
ft_lr = #TODO
ft_max_iters = #TODO
ft_warmup = #TODO
ft_eval_interval = 200
ft_eval_iters = 100


def ft_get_batch(split):
    data = sonnet_train if split == 'train' else sonnet_val
    ix = torch.randint(len(data) - block_size, (ft_batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].copy()) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].copy()) for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

def ft_get_lr(it):
    if it < ft_warmup:
        return ft_lr * it / ft_warmup
    if it > ft_max_iters:
        return ft_lr * 0.1
    decay_ratio = (it - ft_warmup) / (ft_max_iters - ft_warmup)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return ft_lr * 0.1 + coeff * (ft_lr - ft_lr * 0.1)

@torch.no_grad()
def ft_estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(ft_eval_iters)
        for k in range(ft_eval_iters):
            X, Y = ft_get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

ft_optimizer = torch.optim.AdamW(model.parameters(), lr=ft_lr, weight_decay=0.1)

ft_train_losses, ft_val_losses, ft_steps = [], [], []
ft_best_val = 1e9

model.train()
t0 = time.time()

for it in range(ft_max_iters + 1):
    lr = ft_get_lr(it)
    for pg in ft_optimizer.param_groups:
        pg['lr'] = lr

    if it % ft_eval_interval == 0:
        losses = ft_estimate_loss()
        ft_train_losses.append(losses['train'])
        ft_val_losses.append(losses['val'])
        ft_steps.append(it)
        marker = ""
        if losses['val'] < ft_best_val:
            ft_best_val = losses['val']
            marker = " *"
        print(f"step {it:4d} | train {losses['train']:.4f} | val {losses['val']:.4f} | lr {lr:.2e}{marker}")

    X, Y = ft_get_batch('train')
    with ctx:
        _, loss = model(X, Y)

    scaler.scale(loss).backward()
    scaler.unscale_(ft_optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(ft_optimizer)
    scaler.update()
    ft_optimizer.zero_grad(set_to_none=True)
    t0 = time.time()

print(f"\nFine-tuning complete. Best val loss: {ft_best_val:.4f}")

#### 2.8 Fine-tuning loss curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ft_steps, ft_train_losses, label='Train Loss', marker='o', markersize=3)
plt.plot(ft_steps, ft_val_losses, label='Val Loss', marker='s', markersize=3)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('nanoGPT Sonnet Fine-tuning: Train vs Val Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### 2.9 Sample sonnets from fine-tuned nanoGPT

Generate sonnets using the fine-tuned model. Use the `@` marker in the prompt to trigger sonnet-style output.

In [ ]:
print(generate_text("@\nShall I compare thee to a summer's day?\n", max_new_tokens=#TODO
                    , temperature=#TODO
                    ))

---
### 3. Pretrained GPT-2 (124M parameters)

We load [nishantup/nanogpt-slm-124m](https://huggingface.co/nishantup/nanogpt-slm-124m), a 124M-parameter nanoGPT model trained on 133 classic English fiction novels.

Why use a pretrained model? Parts 1 and 2 trained from scratch on ~1M characters. This model has already been trained on vastly more data, so it has a strong general understanding of English grammar, vocabulary, and style. Fine-tuning it on Shakespeare sonnets requires much less data and fewer iterations to produce high-quality output.

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import math
import time
from dataclasses import dataclass
from contextlib import nullcontext

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

#### 3.1 Model

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.2
    bias: bool = False

In [ ]:
# Custom LayerNorm that supports optional bias (PyTorch's built-in always has bias)
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Single linear projects input to Q, K, V concatenated (3x width)
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # Use PyTorch's fused SDPA kernel when available (faster, memory-efficient)
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            # Fallback: manual attention with a causal (lower-triangular) mask
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                         .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        # Split the 3*n_embd projection into separate Q, K, V
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # Reshape into (B, n_head, T, head_dim) for multi-head attention
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            # Flash attention: fused kernel, no explicit mask needed (is_causal=True)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                    dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # Manual scaled dot-product attention with causal mask
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        # Recombine heads and project back to n_embd
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    """Position-wise feedforward: expand 4x, GELU, project back."""
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    """Pre-norm Transformer block: LN → Attention → residual, LN → MLP → residual."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # ModuleDict mirrors GPT-2's naming so we can load HF checkpoints
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),   # token embeddings
            wpe=nn.Embedding(config.block_size, config.n_embd),   # learned positional embeddings
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, bias=config.bias),      # final layer norm
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight tying: share embedding and output projection weights
        # (reduces params and improves generalization — standard in GPT-2)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        # Combine token + positional embeddings, then apply dropout
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            # Training: compute logits for all positions and return loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # Inference: only compute logits for the last position (saves memory)
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            # Crop context to block_size if it's grown too long
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            # Scale logits by temperature before softmax
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                # Zero out everything below the top-k threshold
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
from huggingface_hub import hf_hub_download
import torch

torch.cuda.empty_cache()

model_path = hf_hub_download(
    repo_id="nishantup/nanogpt-slm-124m",
    filename="nanogpt_slm_best.pth"
)

state_dict = torch.load(model_path)

In [ ]:
config = GPTConfig()
model = GPT(config)

model.load_state_dict(state_dict, strict=False)

model = model.to(DEVICE)

print(f"Pretrained nanoGPT loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Config: block_size={config.block_size}, vocab_size={config.vocab_size}, "
      f"n_layer={config.n_layer}, n_head={config.n_head}, n_embd={config.n_embd}")

#### 3.2 Fine-tune on Shakespeare Sonnets

We use OpenAI's GPT-2 BPE tokenizer (`tiktoken`) since that's what the pretrained model expects.


In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
encode = lambda s: enc.encode(s, allowed_special=set())
decode = lambda l: enc.decode(l)
print(f"GPT-2 BPE vocab: {enc.n_vocab} tokens")
print(f"Sonnet text encodes to {len(encode(sonnet_text_raw)):,} tokens "
      f"(from {len(sonnet_text_raw):,} chars, ~{len(sonnet_text_raw)/len(encode(sonnet_text_raw)):.1f} chars/token)")

GPT-2 BPE vocab: 50257 tokens
Sonnet text encodes to 25,535 tokens (from 94,903 chars, ~3.7 chars/token)


Hyperparameters

This model is already very capable, so fine-tuning requires care to avoid catastrophic forgetting or overfitting on the small sonnet corpus.

| Parameter | recommended | Notes |
|-----------|------------------|-------|
| `block_size` | 128–256 | Context window. The pretrained model supports up to 256. Using 256 lets the model see almost an entire sonnet at once. |
| `batch_size` | 4–16 | Smaller batches are fine given the small dataset. 16 is a safe default. |
| `max_iters` | 200–500 | The dataset is tiny — you only need a few hundred steps. Watch val loss for the sweet spot. |
| `learning_rate` | 1e-5 – 5e-4 | Much lower than training from scratch! The pretrained weights are already good; large LR will destroy them. Start around 5e-5 to 1e-4. |

Tip: Because the model already knows English, you should see the loss drop quickly in the first ~50 steps. If it doesn't budge, your LR is too low; if it immediately spikes, your LR is too high.

In [ ]:
block_size = #TODO
batch_size = #TODO
max_iters = #TODO
learning_rate = #TODO
eval_interval = 50
eval_iters = 20
grad_clip = 1.0

In [ ]:
ctx = torch.amp.autocast(device_type='cuda', dtype=ptdtype) if DEVICE_TYPE == 'cuda' else nullcontext()

data = np.array(encode(sonnet_text_raw), dtype=np.int64)
n = len(data)
train_data = data[:int(n * 0.9)]
val_data = data[int(n * 0.9):]
print(f"Fine-tune data (sonnets): {n:,} BPE tokens | train {len(train_data):,} | val {len(val_data):,}")

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].copy()) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].copy()) for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            with ctx:
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

train_data_losses = []
val_data_losses = []
steps = []
best_val = 1e9

model.train()
t0 = time.time()

for it in range(max_iters + 1):
    if it % eval_interval == 0:
        losses = estimate_loss()
        train_data_losses.append(losses['train'])
        val_data_losses.append(losses['val'])
        steps.append(it)
        marker = " *" if losses['val'] < best_val else ""
        if losses['val'] < best_val:
            best_val = losses['val']
        print(f"step {it:4d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}{marker}")

    x, y = get_batch('train')
    with ctx:
        _, loss = model(x, y)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    if DEVICE_TYPE == 'tpu':
        xm.optimizer_step(optimizer)
    else:
        optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    t0 = time.time()

print(f"Fine-tuning complete. Best val loss: {best_val:.4f}")

#### 3.3 Fine-tuning loss curve

Watch for overfitting. If the val loss starts rising while train loss continues to drop, you've gone too far (tweak `max_iters` or `learning_rate`).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(steps, train_data_losses, label='Train Loss', marker='o', markersize=3)
plt.plot(steps, val_data_losses, label='Val Loss', marker='s', markersize=3)
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('GPT-2 Fine-tuning: Train vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train loss: {train_data_losses[-1]:.4f}")
print(f"Final val loss:   {val_data_losses[-1]:.4f}")

#### 3.4 Sample poems

Use the `@` marker as part of the prompt to trigger sonnet-style output.

In [ ]:
def generate(prompt,
             max_new_tokens,
             temperature,
             top_k
             ):
    model.eval()
    tokens = encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=DEVICE)[None, ...]
    with torch.no_grad():
        with ctx:
            y = model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(y[0].tolist())

In [ ]:
print(generate("@\nShall I compare thee to a summer's day?\n",
               max_new_tokens=#TODO,
               temperature=#TODO,
               top_k=#TODO
               ))